# Kurs: [Docker und Kubernetes – Übersicht und Einsatz ](https://www.digicomp.ch/trends/docker-trainings/docker-und-kubernetes-uebersicht-und-einsatz)

Dieses Notebook dient als strukturiertes Hilfsmittel zur Analyse und zum Troubleshooting der Kursumgebung.

Es enthält vorbereitete Abfragen und Korrekturschritte, mit denen typische Probleme (z. B. Deployment-Fehler, Ressourcenengpässe oder Konfigurationsabweichungen) systematisch identifiziert und behoben werden können.


---

### Kein Zugriff auf Kubernetes Cluster

Bei fehlendem Zugriff auf ein **MicroK8s**-Cluster liegt die Ursache häufig in einer fehlenden oder veralteten Kubernetes-Konfiguration.

In diesem Fall kann die Konfiguration neu erzeugt und lokal gespeichert werden:

In [ ]:
%%bash
sudo microk8s config | tee ~/.kube/config

In [ ]:
%%bash
kubectl get nodes -o wide

---

### Server-IP

Die nachfolgende Zelle ermittelt automatisch die IP-Adresse oder den DNS-Namen des aktuellen Servers.

Unterstützt werden sowohl lokale Umgebungen (z. B. Hyper-V, Multipass) als auch Cloud-Plattformen wie AWS, Azure und Google Cloud. Abhängig von der erkannten Umgebung wird die passende Methode zur Ermittlung verwendet.

Wird keine bekannte Umgebung erkannt, erfolgt die Bestimmung der IP-Adresse über die aktive Netzwerkschnittstelle des Servers.

Das Ergebnis (IP-Adresse oder DNS-Name) wird in der Datei `~/data/server-ip` gespeichert und kann von anderen Notebooks weiterverwendet werden.


In [ ]:
%%bash
# Public IP anhand Cloud Provider setzen, WireGuard ueberschreibt alle
curl -fsSL https://raw.githubusercontent.com/mc-b/lerncloud/main/scripts/get-server-ip.sh | bash | tee ~/data/server-ip

- - -

### UUID

Die nachfolgende Zeile generiert eine eindeutige UUID (Universally Unique Identifier).

Die UUID dient als unverwechselbare Kennung, die sicherstellt, dass jede Installation eindeutig identifiziert werden kann. 

Damit kann, z.B. für die IoT Übungen ein eindeutiges MQTT Topic bereitgestellt werden.

Die generierte UUID wird im Python-Dateiformat uuid.py im Verzeichnis ~/data/ gespeichert und wird in den Python-Skripten weiterverwendet.

In [ ]:
%%bash
echo "UUID=\"$(uuid)\"" | tee ~/data/uuid.py

- - -

### Docker.io Login 

Um die Pull-Raten-Limits von Docker zu umgeben, Melden wir uns bei Docker.io an und hinterlegen die Anmeldeinformationen in Kubernetes.


In [ ]:
%%bash
for ns in $(microk8s kubectl get ns --no-headers -o custom-columns=":metadata.name"); do
  microk8s kubectl create secret docker-registry regcred \
    --docker-server=https://index.docker.io/v1/ \
    --docker-username=misegr \
    --docker-password='dckr_pat_Su9Wt8rqpxC2Wixm7_token1234' \
    --namespace $ns
done
for ns in $(microk8s kubectl get ns --no-headers -o custom-columns=":metadata.name"); do
  microk8s kubectl patch serviceaccount default \
    -p '{"imagePullSecrets":[{"name":"regcred"}]}' \
    -n $ns
done


- - - 

### Pull-Raten-Limits

Docker Hub verwendet IP-Adressen, um die Benutzer zu authentifizieren, und Pull-Raten-Limits basieren auf einzelnen IP-Adressen. 

Für **anonyme Benutzer** ist das Ratenlimit auf 100 Abrufe pro 6 Stunden pro IP-Adresse festgelegt.

Die aktuellen Zugriff können wir wie folgt abfragen:

In [ ]:
%%bash
TOKEN=$(curl -s "https://auth.docker.io/token?service=registry.docker.io&scope=repository:ratelimitpreview/test:pull" | jq -r .token)
curl --head -H "Authorization: Bearer $TOKEN" https://registry-1.docker.io/v2/ratelimitpreview/test/manifests/latest

---

### Holen der Container Images von einer Alternativen Container Registry


In [ ]:
%%bash
#!/bin/bash
# pull-alternative-images.sh
# Lädt alternative Images, taggt sie als Ersatz für docker.io

declare -A map=(
  ["docker.io/calico/cni:v3.28.1"]="quay.io/calico/cni:v3.28.1"
  ["docker.io/calico/kube-controllers:v3.28.1"]="quay.io/calico/kube-controllers:v3.28.1"
  ["docker.io/calico/node:v3.28.1"]="quay.io/calico/node:v3.28.1"
  ["docker.io/coredns/coredns:1.10.1"]="registry.k8s.io/coredns/coredns:1.10.1"
  ["docker.io/istio/pilot:1.24.2"]="gcr.io/istio-release/pilot:1.24.2"
  ["docker.io/istio/proxyv2:1.24.2"]="gcr.io/istio-release/proxyv2:1.24.2"
  ["docker.io/kubernetesui/dashboard:v2.7.0"]="registry.gitlab.com/mc-b/misegr/kubernetesui/dashboard:v2.7.0"
  ["docker.io/openzipkin/zipkin-slim:3.4.0"]="ghcr.io/openzipkin/zipkin-slim:3.4.0"
)

for orig in "${!map[@]}"; do
  alt="${map[$orig]}"
  echo "Pulling alternative: $alt"
  microk8s ctr images pull "$alt"
  echo "Tagging as $orig"
  microk8s ctr images tag "$alt" "$orig" || true
done
